# Personalized Content Recommendation & Affinity Engine
## Visual Exploratory Data Analysis, Model Benchmarks & Lift Analysis

This notebook analyzes the multi-modal implicit interaction telemetry, compares latent collaborative filtering vs semantic topic embeddings, and benchmarks Top-K ranking performance (Recall@K, NDCG@K, MRR).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure project root is in path
sys.path.insert(0, os.path.abspath(".."))

from src.database import DatabaseManager
from src.telemetry_processor import TelemetryProcessor
from src.models.collaborative import TwoTowerRecommender
from src.models.content_based import ContentBasedRecommender
from src.models.hybrid_engine import HybridRecommendationEngine
from src.evaluate import BenchmarkEvaluator

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
%matplotlib inline
print("Libraries loaded successfully.")

## 1. Implicit Interaction Telemetry & Composite Affinity Distribution
Here we examine the distribution of implicit user signals: dwell duration, scroll depth, and the formulated composite affinity score.

$$\text{Affinity} = w_1 \cdot \text{click} + w_2 \cdot \text{norm\_dwell} + w_3 \cdot \text{scroll\_depth}$$

In [2]:
db_path = os.path.abspath("../data/content_rec.db")
db_mgr = DatabaseManager(db_url=f"sqlite:///{db_path}")

with db_mgr.get_session() as session:
    from src.database import InteractionModel
    interactions = [i.to_dict() for i in session.query(InteractionModel).limit(5000).all()]

df_telemetry = pd.DataFrame(interactions)
print(f"Loaded {len(df_telemetry)} telemetry records for exploratory analysis.")
display(df_telemetry.head() if "display" in dir() else print(df_telemetry.head()))

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Dwell Duration Distribution
axes[0].hist(df_telemetry["dwell_time"], bins=35, color="#2b5c8f", edgecolor="white", alpha=0.85)
axes[0].set_title("Dwell Time Distribution (seconds)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Dwell Time (s)")
axes[0].set_ylabel("Frequency")

# 2. Scroll Depth Traversal
axes[1].hist(df_telemetry["scroll_depth"], bins=25, color="#2a9d8f", edgecolor="white", alpha=0.85)
axes[1].set_title("Scroll Depth Traversal ([0.0, 1.0])", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Scroll Depth %")

# 3. Formulated Composite Affinity Target
axes[2].hist(df_telemetry["affinity_score"], bins=30, color="#e76f51", edgecolor="white", alpha=0.85)
axes[2].set_title("Composite Affinity Target Score", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Affinity Score [0.0 - 1.0]")

plt.tight_layout()
plt.show()

## 2. Quantitative Model Benchmark & Lift Analysis
Evaluating Top-K ranking performance across **Popularity Baseline**, **Content-Based Semantic Embeddings**, **Two-Tower Neural CF**, and the **Full Hybrid Engine**.

In [4]:
cf_ckpt = os.path.abspath("../checkpoints/two_tower_best.pt")
cf_model = TwoTowerRecommender.load_checkpoint(cf_ckpt)

emb_cache = os.path.abspath("../checkpoints/content_embeddings.npz")
loaded = np.load(emb_cache)
content_model = ContentBasedRecommender({k: loaded[k] for k in loaded.files})

hybrid_engine = HybridRecommendationEngine(
    collaborative_model=cf_model,
    content_model=content_model,
    db_manager=db_mgr,
    alpha_cf=0.60,
    beta_category=0.10
)

popular_articles = db_mgr.get_popular_articles(limit=100)
popularity_ranking = [p["article_id"] for p in popular_articles]

evaluator = BenchmarkEvaluator(db_mgr, hybrid_engine, popularity_ranking)
df_benchmark = evaluator.run_benchmark(k_list=[5, 10])
display(df_benchmark if "display" in dir() else print(df_benchmark))

In [5]:
# Visual comparison plot for Recall@5, Recall@10, and NDCG@10
models = df_benchmark["Model"].tolist()
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
rects1 = ax.bar(x - width, df_benchmark["Recall@5"], width, label="Recall@5", color="#457b9d")
rects2 = ax.bar(x, df_benchmark["Recall@10"], width, label="Recall@10", color="#1d3557")
rects3 = ax.bar(x + width, df_benchmark["NDCG@10"], width, label="NDCG@10", color="#e63946")

ax.set_ylabel("Metric Score", fontsize=12)
ax.set_title("Recommendation Benchmark: Baseline vs. Content vs. CF vs. Hybrid", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.legend(frameon=True, fontsize=11)

# Value labels
for rect in [rects1, rects2, rects3]:
    for bar in rect:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()